# City-Day Air Quality Index (AQI) Prediction

This notebook builds and evaluates reproducible regression pipelines for estimating a city-day AQI from same-day pollutant measurements, city, and calendar fields. It expects `city_day.csv` in the notebook's working directory. The chronological holdout evaluates later dates; this is an AQI estimation task, not a forecast made before same-day pollutant measurements are available.

## 1. Project Overview and Problem Definition

The target is `AQI`, a numeric field. `AQI_Bucket` is excluded because it is a category associated with the AQI and could disclose target information. The inputs are city, date-derived calendar features, and pollutant columns. We measure generalization to dates later than the training period, select models using time-series cross-validation on training data, and report the untouched final period once. MAE is the primary selection metric because it expresses typical error in AQI points and is less sensitive to a few extreme errors than squared error. R² and RMSE are supporting metrics.

The dataset provides no accompanying data dictionary, measurement units, AQI calculation formula, or collection methodology. Descriptions below are limited to what field names and observed values establish. Missing target rows cannot train or score a supervised model. City/date coverage and pollutant availability may vary over time, so future deployment requires monitoring for drift and data-quality changes.

## 2. Imports and Environment Setup

Install the required packages if needed: `pip install pandas numpy matplotlib seaborn scikit-learn joblib`. No optional boosting package is required. A fixed seed is used where algorithms are stochastic.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42
sns.set_theme(style='whitegrid', context='notebook')
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

## 3. Dataset Loading

Load the supplied file and parse the date column. Invalid date values are reported; they are excluded from modeling because calendar features and chronological splitting require valid dates.

In [ ]:
DATA_PATH = Path('city_day.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place city_day.csv in the notebook working directory.')
df = pd.read_csv(DATA_PATH)
if 'Date' not in df or 'AQI' not in df:
    raise ValueError('Expected Date and AQI columns were not found.')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:\n', df.dtypes)
display(df.head())
display(df.describe(include='all').T)

## 4. Dataset Metadata

The source file contains names but no authoritative definitions or units. The descriptions are deliberately conservative; confirm operational meaning with the data provider before deployment.

In [ ]:
known_notes = {
    'City': 'City label in the source file; geographic coordinate or administrative definition not supplied.',
    'Date': 'Calendar date represented by the row.',
    'AQI': 'Numeric AQI target as named by the source; scale and computation method not supplied.',
    'AQI_Bucket': 'Categorical AQI bucket as named by the source; thresholds not supplied.'
}
for col in df.columns:
    if col not in known_notes:
        known_notes[col] = ('Pollutant measurement field inferred from its label; units and measurement method are not supplied.'
                            if col not in ['Date', 'City'] else '')
metadata_rows = []
for col in df.columns:
    s = df[col]
    examples = s.dropna().astype(str).drop_duplicates().head(4).tolist()
    metadata_rows.append({
        'Feature': col, 'Data type': str(s.dtype), 'Description': known_notes[col],
        'Missing': int(s.isna().sum()), 'Unique': int(s.nunique(dropna=True)),
        'Examples': ', '.join(examples),
        'Summary': (str(s.describe().to_dict()) if pd.api.types.is_numeric_dtype(s) else '')
    })
metadata = pd.DataFrame(metadata_rows)
display(metadata)

## 5. Data Quality Assessment

Review missing values, repeated rows, target coverage, and categorical values before defining the modeling sample.

In [ ]:
print('Duplicate rows:', int(df.duplicated().sum()))
print('Invalid dates:', int(df['Date'].isna().sum()))
print('Rows with missing AQI:', int(df['AQI'].isna().sum()))
display(df.isna().sum().sort_values(ascending=False).rename('missing').to_frame())
for col in ['City', 'AQI_Bucket']:
    if col in df:
        print(f'\n{col} values (top 20):')
        display(df[col].value_counts(dropna=False).head(20).to_frame('count'))

## 6. Exploratory Data Analysis

The charts below summarize observed distributions and relationships. Interpretations are printed from computed values instead of being prefilled with unverified conclusions.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, ax = plt.subplots(figsize=(10, 4))
df.isna().mean().sort_values(ascending=False).plot.bar(ax=ax, color='steelblue')
ax.set(title='Fraction of values missing by feature', ylabel='Fraction missing', xlabel='Feature'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
print('Interpretation: bars show the proportion missing; high-missingness fields contribute less information and will require imputation.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=df, x='AQI', bins=40, kde=True, ax=ax)
ax.set(title='Distribution of the AQI target', xlabel='AQI', ylabel='City-day count'); plt.tight_layout(); plt.show()
print('Observed AQI summary:', df['AQI'].describe().round(2).to_dict())

In [ ]:
plot_numeric = [c for c in numeric_cols if c != 'AQI']
if plot_numeric:
    df[plot_numeric].hist(figsize=(15, 12), bins=30, grid=False, color='slateblue')
    plt.suptitle('Distributions of numeric measurements', y=1.01); plt.tight_layout(); plt.show()
print('Interpretation: inspect skew and extreme tails before choosing transformations; the baseline pipelines use median imputation and do not discard measured extremes.')

In [ ]:
if plot_numeric:
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=df[plot_numeric], orient='h', ax=ax, showfliers=False)
    ax.set(title='Numeric measurement ranges (fliers hidden for readability)', xlabel='Recorded value'); plt.tight_layout(); plt.show()
print('Interpretation: boxes and whiskers aid outlier investigation; extreme values are retained unless data validation establishes an error.')

In [ ]:
corr_cols = [c for c in numeric_cols if df[c].notna().sum() > 1]
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), cmap='vlag', center=0, ax=ax)
ax.set(title='Correlation among numeric fields'); plt.tight_layout(); plt.show()
print('Interpretation: correlations are descriptive, can reflect shared construction or temporal patterns, and do not establish causation.')

In [ ]:
if 'City' in df:
    city_counts = df['City'].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(11, 5)); city_counts.sort_values().plot.barh(ax=ax, color='teal')
    ax.set(title='Records by city (top 20)', xlabel='Rows', ylabel='City'); plt.tight_layout(); plt.show()
    print('Cities represented:', df['City'].nunique())
if 'AQI_Bucket' in df:
    fig, ax = plt.subplots(figsize=(9, 4)); df['AQI_Bucket'].value_counts(dropna=False).plot.bar(ax=ax, color='darkorange')
    ax.set(title='Source AQI_Bucket counts (missing included)', xlabel='AQI_Bucket', ylabel='Rows'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
    print('AQI_Bucket is excluded from predictors because it is associated with the AQI target.')

In [ ]:
valid_target = df.dropna(subset=['AQI', 'Date']).copy()
year_aqi = valid_target.assign(Year=valid_target['Date'].dt.year).groupby('Year')['AQI'].agg(['count','mean','median'])
display(year_aqi)
fig, ax = plt.subplots(figsize=(11, 4)); sns.lineplot(data=valid_target, x='Date', y='AQI', errorbar=None, ax=ax)
ax.set(title='Observed AQI over time (all cities pooled)', xlabel='Date', ylabel='AQI'); plt.tight_layout(); plt.show()
print('Interpretation: pooled observations may change with city composition and missingness; the chronological split avoids training on later dates.')

## 7. Data Preprocessing and Feature Engineering

Extract calendar components from `Date` before splitting because these are deterministic properties of each record. The fitted imputers, scaling, and city encoder remain inside each pipeline and are learned from training folds only. Exact duplicate records are dropped. No outlier clipping is applied without source-specific validation. `AQI_Bucket` is never an input.

In [ ]:
data = df.drop_duplicates().copy()
data = data.dropna(subset=['AQI', 'Date']).sort_values(['Date', 'City']).reset_index(drop=True)
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['DayOfWeek'] = data['Date'].dt.dayofweek
data['DayOfYear'] = data['Date'].dt.dayofyear
data['IsWeekend'] = (data['DayOfWeek'] >= 5).astype(int)
target = data['AQI'].astype(float)
feature_cols = [c for c in data.columns if c not in ['AQI', 'AQI_Bucket', 'Date']]
X = data[feature_cols].copy()
categorical_features = [c for c in X.columns if X[c].dtype == 'object' or isinstance(X[c].dtype, pd.CategoricalDtype)]
numeric_features = [c for c in X.columns if c not in categorical_features]
print('Modeling rows:', len(X), '| numeric:', numeric_features, '| categorical:', categorical_features)

## 8. Chronological Train-Test Split

Rows are ordered by date. All records on the latest 20% of distinct dates form the held-out test period. This evaluates future-date performance while retaining all cities and records that share a date on the same side of the split.

In [ ]:
unique_dates = np.sort(data['Date'].unique())
cutoff_idx = max(1, int(np.floor(len(unique_dates) * 0.8)))
cutoff_date = pd.Timestamp(unique_dates[cutoff_idx]) if cutoff_idx < len(unique_dates) else pd.Timestamp(unique_dates[-1])
train_mask = data['Date'] < cutoff_date
X_train, X_test = X.loc[train_mask].copy(), X.loc[~train_mask].copy()
y_train, y_test = target.loc[train_mask].copy(), target.loc[~train_mask].copy()
print('Cutoff:', cutoff_date, '| train:', X_train.shape, '| test:', X_test.shape)
print('Train dates:', data.loc[train_mask, 'Date'].min(), 'to', data.loc[train_mask, 'Date'].max())
print('Test dates:', data.loc[~train_mask, 'Date'].min(), 'to', data.loc[~train_mask, 'Date'].max())
if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError('Chronological split produced an empty partition.')

## 9. Baseline Model

The baseline predicts the training-set median AQI for every record. It provides a reference for judging whether feature-based models add value.

In [ ]:
baseline_prediction = np.full(len(y_test), y_train.median())
baseline_metrics = {'Model':'Training median baseline', 'MAE':mean_absolute_error(y_test, baseline_prediction), 'RMSE':mean_squared_error(y_test, baseline_prediction) ** 0.5, 'R2':r2_score(y_test, baseline_prediction)}
display(pd.DataFrame([baseline_metrics]))


## 10. Model Implementation and Training

Each estimator receives the same chronological split and fold-local preprocessing. Six distinct regressors are included; Random Forest and Extra Trees are ensembles. Scaling is included for comparable pipelines and benefits distance/linear models; tree estimators do not depend on it. A dense one-hot representation supports HistGradientBoosting on this modest number of cities.

In [ ]:
numeric_preprocessor = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_preprocessor = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_preprocessor, numeric_features), ('cat', categorical_preprocessor, categorical_features)], remainder='drop')
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=18, min_samples_leaf=5, random_state=SEED),
    'Random Forest': RandomForestRegressor(n_estimators=160, min_samples_leaf=2, n_jobs=1, random_state=SEED),
    'Extra Trees': ExtraTreesRegressor(n_estimators=160, min_samples_leaf=2, n_jobs=1, random_state=SEED),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=140, learning_rate=0.08, max_leaf_nodes=20, l2_regularization=1.0, random_state=SEED),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=12, weights='distance', n_jobs=1),
}
fitted_models, model_records = {}, []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', clone(preprocessor)), ('model', clone(estimator))])
    start = time.perf_counter(); pipe.fit(X_train, y_train); fit_seconds = time.perf_counter() - start
    start = time.perf_counter(); pred = pipe.predict(X_test); predict_seconds = time.perf_counter() - start
    fitted_models[name] = pipe
    model_records.append({'Model':name, 'MAE':mean_absolute_error(y_test, pred), 'RMSE':mean_squared_error(y_test, pred)**0.5, 'R2':r2_score(y_test, pred), 'Fit seconds':fit_seconds, 'Predict seconds':predict_seconds})
results = pd.DataFrame(model_records).sort_values('MAE')
display(results)

## 11. Cross-Validation and Model Comparison

TimeSeriesSplit preserves temporal order within the training period. The chronological folds estimate how model errors vary over later training periods. The final test set is not used for selection; it is shown above only as the required initial comparison and will be evaluated after tuning below.

In [ ]:
n_splits = min(4, max(2, len(X_train) // 1000))
tscv = TimeSeriesSplit(n_splits=n_splits)
cv_records = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', clone(preprocessor)), ('model', clone(estimator))])
    scores = -cross_val_score(pipe, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=1)
    cv_records.append({'Model':name, 'CV MAE mean':scores.mean(), 'CV MAE std':scores.std()})
cv_results = pd.DataFrame(cv_records).sort_values('CV MAE mean')
results = results.merge(cv_results, on='Model').sort_values('CV MAE mean')
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5)); sns.barplot(data=results, x='CV MAE mean', y='Model', ax=ax, color='cornflowerblue')
ax.set(title='Training-period time-series CV comparison (lower MAE is better)', xlabel='Mean validation MAE (AQI points)', ylabel=''); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9, 4)); sns.barplot(data=results, x='Fit seconds', y='Model', ax=ax, color='seagreen')
ax.set(title='Fit time on the chronological training partition', xlabel='Seconds', ylabel=''); plt.tight_layout(); plt.show()
print('Interpretation: compare validation MAE and its fold variability before choosing a model; runtime is a practical tradeoff, not the selection metric.')

## 12. Hyperparameter Tuning

Tune the promising HistGradientBoosting model with a small randomized search and chronological folds. Search decisions use only the training partition. The selected estimator is refit by the search on all training rows.

In [ ]:
tuning_pipe = Pipeline([('preprocess', clone(preprocessor)), ('model', HistGradientBoostingRegressor(random_state=SEED))])
param_distributions = {
    'model__learning_rate':[0.04, 0.08, 0.12],
    'model__max_iter':[100, 180, 260],
    'model__max_leaf_nodes':[15, 25, 35],
    'model__l2_regularization':[0.0, 1.0, 5.0]
}
search = RandomizedSearchCV(tuning_pipe, param_distributions=param_distributions, n_iter=10, scoring='neg_mean_absolute_error', cv=tscv, random_state=SEED, n_jobs=1, refit=True, verbose=1)
search.fit(X_train, y_train)
print('Best CV MAE:', -search.best_score_)
print('Best parameters:', search.best_params_)

## 13. Final Model Selection and Untouched Test Evaluation

Select the tuned candidate if its training-only cross-validation MAE beats the best untuned model's cross-validation MAE; otherwise use the best untuned model. Only now score the selected pipeline on the held-out later-date period.

In [ ]:
best_untuned_name = cv_results.iloc[0]['Model']
best_untuned_cv_mae = float(cv_results.iloc[0]['CV MAE mean'])
tuned_cv_mae = -float(search.best_score_)
if tuned_cv_mae < best_untuned_cv_mae:
    selected_name, selected_pipeline = 'Tuned Hist Gradient Boosting', search.best_estimator_
else:
    selected_name, selected_pipeline = best_untuned_name, fitted_models[best_untuned_name]
selected_prediction = selected_pipeline.predict(X_test)
final_metrics = {'Selected model':selected_name, 'Selection CV MAE':min(tuned_cv_mae, best_untuned_cv_mae), 'Test MAE':mean_absolute_error(y_test, selected_prediction), 'Test RMSE':mean_squared_error(y_test, selected_prediction)**0.5, 'Test R2':r2_score(y_test, selected_prediction)}
display(pd.DataFrame([final_metrics]))
print('Untuned CV leader:', best_untuned_name, '| untuned CV MAE:', best_untuned_cv_mae, '| tuned CV MAE:', tuned_cv_mae)

## 14. Explainability and Diagnostics

Permutation importance measures the increase in held-out MAE after shuffling each feature. It is a model-agnostic diagnostic, but correlated inputs can share importance and later-period importance may not represent all cities or future regimes. Actual-versus-predicted and residual plots expose systematic errors.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6)); sns.scatterplot(x=y_test, y=selected_prediction, alpha=0.35, ax=ax)
limits = [min(y_test.min(), selected_prediction.min()), max(y_test.max(), selected_prediction.max())]
ax.plot(limits, limits, '--', color='red'); ax.set(title=f'Actual vs predicted - {selected_name}', xlabel='Actual AQI', ylabel='Predicted AQI'); plt.tight_layout(); plt.show()
residuals = y_test.to_numpy() - selected_prediction
fig, axes = plt.subplots(1, 2, figsize=(12, 4)); sns.histplot(residuals, kde=True, ax=axes[0]); axes[0].set(title='Residual distribution', xlabel='Actual minus predicted AQI')
sns.scatterplot(x=selected_prediction, y=residuals, alpha=0.35, ax=axes[1]); axes[1].axhline(0, color='red', linestyle='--'); axes[1].set(title='Residuals vs predicted', xlabel='Predicted AQI', ylabel='Residual'); plt.tight_layout(); plt.show()
print('Interpretation: assess calibration around the diagonal, residual centering, changing residual spread, and systematic patterns.')

In [ ]:
importance = permutation_importance(selected_pipeline, X_test, y_test, scoring='neg_mean_absolute_error', n_repeats=5, random_state=SEED, n_jobs=1)
importance_table = pd.DataFrame({'Feature':feature_cols, 'MAE increase':importance.importances_mean, 'Std':importance.importances_std}).sort_values('MAE increase', ascending=False)
display(importance_table.head(15))
fig, ax = plt.subplots(figsize=(9, 5)); sns.barplot(data=importance_table.head(15), x='MAE increase', y='Feature', ax=ax, color='mediumpurple')
ax.set(title='Permutation importance on held-out dates', xlabel='Increase in MAE when shuffled (AQI points)', ylabel=''); plt.tight_layout(); plt.show()

## 15. Model Serialization

Save the fitted preprocessing-and-model pipeline as one joblib artifact, along with a compact JSON metadata file describing target and expected input schema. The artifact is written to the current working directory.

In [ ]:
ARTIFACT_PATH = Path('aqi_prediction_pipeline.joblib')
METADATA_PATH = Path('aqi_prediction_metadata.json')
joblib.dump(selected_pipeline, ARTIFACT_PATH)
metadata_payload = {'target':'AQI', 'excluded_leakage_field':'AQI_Bucket', 'input_features':feature_cols, 'numeric_features':numeric_features, 'categorical_features':categorical_features, 'date_features':['Year','Month','Day','DayOfWeek','DayOfYear','IsWeekend'], 'training_cutoff_exclusive':str(cutoff_date.date()), 'selected_model':selected_name, 'selection_metric':'TimeSeriesSplit cross-validation MAE'}
METADATA_PATH.write_text(pd.Series(metadata_payload).to_json(indent=2), encoding='utf-8')
loaded_pipeline = joblib.load(ARTIFACT_PATH)
print('Saved:', ARTIFACT_PATH.resolve(), METADATA_PATH.resolve())
print('Reload prediction sample:', loaded_pipeline.predict(X_test.head(3)))

## 16. Prediction Demonstration

This helper validates required columns, reports unexpected columns, checks that required numeric fields can be interpreted numerically, and aligns the input schema. Missing values are passed to the saved imputers; preprocessing is not refitted during inference.

In [ ]:
def predict_aqi(new_records, pipeline=loaded_pipeline):
    """Predict AQI from a DataFrame with the trained model input schema."""
    if not isinstance(new_records, pd.DataFrame):
        raise TypeError('new_records must be a pandas DataFrame.')
    missing = sorted(set(feature_cols) - set(new_records.columns))
    extra = sorted(set(new_records.columns) - set(feature_cols))
    if missing:
        raise ValueError(f'Missing required input columns: {missing}')
    if extra:
        raise ValueError(f'Unexpected input columns: {extra}')
    aligned = new_records.loc[:, feature_cols].copy()
    for col in numeric_features:
        aligned[col] = pd.to_numeric(aligned[col], errors='coerce')
    return pipeline.predict(aligned)

demo = X_test.head(5).copy()
display(pd.DataFrame({'Predicted AQI':predict_aqi(demo), 'Actual AQI':y_test.loc[demo.index].to_numpy()}, index=demo.index))

## 17. Technical Conclusions and Limitations

Use the computed comparison tables, validation scores, final later-date metrics, residual plots, and permutation importance above to describe model performance. Do not interpret these estimates as pre-event forecasts: same-day pollutant readings are required inputs. The dataset does not establish measurement units, monitoring-station representativeness, AQI formula, sensor calibration, or missingness mechanism. A temporal holdout is more realistic than a random split for later dates, but records from known cities remain in both partitions; it does not measure generalization to unseen cities. AQI construction may be mathematically linked to pollutant readings, and the exact formula is unavailable. Before deployment, validate source definitions, monitor missingness and drift, evaluate city-specific error, and confirm that input availability matches the intended prediction time.